# CineMatch – Baseline Models & Comparison

**Baselines implemented:**
1. **Global Popularity** — Recommend most-rated movies (no personalization)
2. **ALS Collaborative Filtering** — Matrix factorization on user×item interactions (movielens only)
3. **Content-Based FAISS** — Semantic similarity via IMDB BGE-M3 embeddings
4. **FAISS + BGE-M3 Reranker** — Two-stage: FAISS retrieval → multi-granularity reranking

**Evaluation:** Precision@K, Recall@K, NDCG@K, Hit Rate@K, MRR, Coverage

In [1]:
# Installs

!pip install -q implicit faiss-gpu-cu12 sentence-transformers FlagEmbedding
!pip uninstall transformers tokenizers -y
!pip install "transformers>=4.40.0"

# Local (CPU):
# !pip install -q implicit faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 19.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.9 MB/s eta 0:00:00


In [1]:
from __future__ import annotations

import json, time, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Runtime Detection
try:
    from google.colab import drive
    RUNTIME = 'colab'
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/cinematch')
except ImportError:
    RUNTIME = 'local'
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / 'Data').exists() and (candidate / 'src').exists():
            DATA_DIR = candidate / 'Data'
            break

# ── Paths ──
RATINGS_PATH   = DATA_DIR / 'Data'/ 'ml-32m' / 'ratings.csv'
MOVIES_PATH    = DATA_DIR / 'Data'/ 'ml-32m' / 'movies.csv'
LINKS_PATH     = DATA_DIR / 'Data'/ 'ml-32m' / 'links.csv'
IMDB_META_PATH = DATA_DIR / 'outputs' / 'imdb' / 'imdb_movies_meta.csv'
IMDB_FAISS_PATH = DATA_DIR / 'outputs' / 'imdb' / 'imdb_movies_bge_m3_flatip.faiss'

# TMDB FAISS:
TMDB_FAISS_PATH = DATA_DIR / 'outputs' / 'tmdb_qwen4b.faiss'
TMDB_CATALOG_PATH = DATA_DIR / 'outputs' / 'tmdb_semantic_catalog_alllangs_with_new_movies.csv'

print(f'Runtime: {RUNTIME}')
print(f'Data dir: {DATA_DIR}')
for name, p in [('ratings', RATINGS_PATH), ('movies', MOVIES_PATH), ('links', LINKS_PATH), ('imdb_meta', IMDB_META_PATH), ('imdb_faiss', IMDB_FAISS_PATH)]:
    print(f'  {name}: {"✓" if p.exists() else "✗"} {p.name}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: colab
Data dir: /content/drive/MyDrive/cinematch
  ratings: ✓ ratings.csv
  movies: ✓ movies.csv
  links: ✓ links.csv
  imdb_meta: ✓ imdb_movies_meta.csv
  imdb_faiss: ✓ imdb_movies_bge_m3_flatip.faiss


## Data Loading & Temporal Train/Test Split
Load MovieLens 32M ratings, apply temporal 80/20 split, and sample evaluation users.

In [2]:
# Load Ratings (chunked for memory efficiency)
ratings_dtypes = {'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int32'}
ratings = pd.read_csv(RATINGS_PATH, dtype=ratings_dtypes)
print(f'Total ratings: {len(ratings):,}')

# Load Movies & Links for metadata
movies = pd.read_csv(MOVIES_PATH)
links = pd.read_csv(LINKS_PATH)
links['tconst'] = links['imdbId'].apply(lambda x: f'tt{int(x):07d}' if pd.notna(x) else None)

# Temporal Split: 80/20 by global timestamp
split_ts = int(np.quantile(ratings['timestamp'], 0.80))
train = ratings[ratings['timestamp'] <= split_ts].copy()
test  = ratings[ratings['timestamp'] >  split_ts].copy()
print(f'Train: {len(train):,} | Test: {len(test):,}')
print(f'Split timestamp: {pd.to_datetime(split_ts, unit="s").strftime("%Y-%m-%d")}')

# Positive test items (rating >= 3.5 = "liked")
test_pos = test[test['rating'] >= 3.5]

train_user_items = train.groupby('userId')['movieId'].apply(set).to_dict()
test_user_items  = test_pos.groupby('userId')['movieId'].apply(set).to_dict()

# Sample evaluation users: must have ≥1 positive test item & ≥5 train items
eval_candidates = [
    u for u in test_user_items
    if len(test_user_items[u]) >= 1 and u in train_user_items and len(train_user_items[u]) >= 5
]
N_EVAL = 5000
np.random.seed(42)
eval_users = np.random.choice(eval_candidates, min(N_EVAL, len(eval_candidates)), replace=False)

print(f'Eval users: {len(eval_users):,} (sampled from {len(eval_candidates):,} eligible)')
print(f'Avg positive test items per eval user: {np.mean([len(test_user_items[u]) for u in eval_users]):.1f}')

Total ratings: 32,000,204
Train: 25,600,163 | Test: 6,400,041
Split timestamp: 2018-10-03
Eval users: 5,000 (sampled from 7,654 eligible)
Avg positive test items per eval user: 81.0


## Evaluation Framework
Standard top-K ranking metrics for recommender systems.

In [3]:
K = 20  # Top-K for evaluation

def precision_at_k(recommended: list, relevant: set, k: int = K) -> float:
    rec = recommended[:k]
    if not rec:
        return 0.0
    return len(set(rec) & relevant) / k

def recall_at_k(recommended: list, relevant: set, k: int = K) -> float:
    rec = recommended[:k]
    if not relevant:
        return 0.0
    return len(set(rec) & relevant) / len(relevant)

def ndcg_at_k(recommended: list, relevant: set, k: int = K) -> float:
    rec = recommended[:k]
    dcg = sum(1.0 / np.log2(i + 2) for i, m in enumerate(rec) if m in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def hit_rate_at_k(recommended: list, relevant: set, k: int = K) -> float:
    return float(len(set(recommended[:k]) & relevant) > 0)

def mrr(recommended: list, relevant: set, k: int = K) -> float:
    for i, m in enumerate(recommended[:k]):
        if m in relevant:
            return 1.0 / (i + 1)
    return 0.0

def evaluate_model(recommend_fn, model_name: str, k: int = K) -> dict:
    """Evaluate a recommender on sampled eval users. recommend_fn(user_id, k) -> list[movieId]"""
    metrics = defaultdict(list)
    all_recs = set()
    t0 = time.time()

    for uid in eval_users:
        relevant = test_user_items[uid]
        recs = recommend_fn(uid, k)
        all_recs.update(recs)

        metrics['precision'].append(precision_at_k(recs, relevant, k))
        metrics['recall'].append(recall_at_k(recs, relevant, k))
        metrics['ndcg'].append(ndcg_at_k(recs, relevant, k))
        metrics['hit_rate'].append(hit_rate_at_k(recs, relevant, k))
        metrics['mrr'].append(mrr(recs, relevant, k))

    elapsed = time.time() - t0
    result = {
        'Model': model_name,
        f'Precision@{k}': np.mean(metrics['precision']),
        f'Recall@{k}': np.mean(metrics['recall']),
        f'NDCG@{k}': np.mean(metrics['ndcg']),
        f'HitRate@{k}': np.mean(metrics['hit_rate']),
        f'MRR@{k}': np.mean(metrics['mrr']),
        'Coverage': len(all_recs),
        'Eval_sec': round(elapsed, 1),
    }
    print(f'{model_name}: P@{k}={result[f"Precision@{k}"]:.4f}  '
          f'NDCG@{k}={result[f"NDCG@{k}"]:.4f}  '
          f'HR@{k}={result[f"HitRate@{k}"]:.4f}  ({elapsed:.1f}s)')
    return result

all_results = []

## Baseline 1: Global Popularity
The simplest "naive" model — recommend the most-rated movies to everyone.
No personalization. This is the floor performance any model must beat.

In [6]:
# Global Popularity: rank movies by # of ratings in training set
movie_popularity = (
    train.groupby('movieId')['rating']
    .agg(count='count', mean_rating='mean')
    .sort_values('count', ascending=False)
)
popular_movies = movie_popularity.index.tolist()

def popularity_recommend(user_id: int, k: int = K) -> list:
    seen = train_user_items.get(user_id, set())
    return [m for m in popular_movies if m not in seen][:k]

print(f'Top 10 popular movies:')
top5 = movie_popularity.head(10).merge(movies[['movieId', 'title']], left_index=True, right_on='movieId')
print(top5[['title', 'count', 'mean_rating']].to_string(index=False))

print('\nEvaluating Global Popularity...')
all_results.append(evaluate_model(popularity_recommend, 'Global Popularity'))

Top 10 popular movies:
                                    title  count  mean_rating
                      Forrest Gump (1994)  85040     4.043532
         Shawshank Redemption, The (1994)  84098     4.416746
                      Pulp Fiction (1994)  83041     4.183596
         Silence of the Lambs, The (1991)  77537     4.151882
                       Matrix, The (1999)  74304     4.149858
Star Wars: Episode IV - A New Hope (1977)  71397     4.128696
                     Jurassic Park (1993)  68451     3.672751
                  Schindler's List (1993)  63076     4.254701
                        Braveheart (1995)  62775     4.003425
        Terminator 2: Judgment Day (1991)  60697     3.941406

Evaluating Global Popularity...
Global Popularity: P@20=0.0933  NDCG@20=0.1113  HR@20=0.5216  (10.5s)


## Baseline 2: ALS Collaborative Filtering
Matrix factorization using **Alternating Least Squares (implicit library)**.
Learns latent user & item factors from the rating matrix. Strong baseline for personalized recommendation.

In [7]:
from implicit.als import AlternatingLeastSquares

# Build sparse user×item matrix
# Map userId/movieId to contiguous indices
user_cat = pd.Categorical(train['userId'])
item_cat = pd.Categorical(train['movieId'], categories=movies['movieId'].unique())

# Confidence-weighted: higher rating -> higher confidence
# confidence = 1 + α * (rating - min_rating)  [standard implicit feedback conversion]
alpha = 2.0
confidence = 1.0 + alpha * (train['rating'].values - 0.5)

user_item_sparse = csr_matrix(
    (confidence.astype(np.float32), (user_cat.codes, item_cat.codes)),
    shape=(len(user_cat.categories), len(item_cat.categories)),
)

# Category lookups for mapping back
user_id_to_idx = {uid: idx for idx, uid in enumerate(user_cat.categories)}
item_id_to_idx = {mid: idx for idx, mid in enumerate(item_cat.categories)}
idx_to_item_id = {idx: mid for mid, idx in item_id_to_idx.items()}

print(f'Sparse matrix: {user_item_sparse.shape} with {user_item_sparse.nnz:,} non-zeros')
print(f'Density: {user_item_sparse.nnz / np.prod(user_item_sparse.shape) * 100:.4f}%')

# Train ALS
als = AlternatingLeastSquares(
    factors=128,
    regularization=0.01,
    iterations=30,
    use_gpu=True,
    random_state=42,
)

# from implicit.bpr import BayesianPersonalizedRanking

# bpr = BayesianPersonalizedRanking(
#     factors=256,
#     learning_rate=0.05,
#     regularization=0.01,
#     iterations=100,
#     use_gpu=True,
#     random_state=42,
# )
# bpr.fit(user_item_sparse)

print('Training ALS...')
t0 = time.time()
als.fit(user_item_sparse)
print(f'ALS trained in {time.time() - t0:.1f}s')

# ALS Recommender
def als_recommend(user_id: int, k: int = K) -> list:
    if user_id not in user_id_to_idx:
        return popularity_recommend(user_id, k)  # cold-start fallback
    uidx = user_id_to_idx[user_id]
    item_ids, scores = als.recommend(uidx, user_item_sparse[uidx], N=k, filter_already_liked_items=True)
    return [idx_to_item_id[i] for i in item_ids if i in idx_to_item_id]

print('\nEvaluating ALS CF...')
all_results.append(evaluate_model(als_recommend, 'ALS Collaborative Filtering'))

Sparse matrix: (170491, 87585) with 25,600,163 non-zeros
Density: 0.1714%
Training ALS...


  0%|          | 0/30 [00:00<?, ?it/s]

ALS trained in 12.0s

Evaluating ALS CF...
ALS Collaborative Filtering: P@20=0.1476  NDCG@20=0.1698  HR@20=0.6782  (3.8s)


## Baseline 3: Content-Based (FAISS Semantic Search)
Uses the prebuilt IMDB BGE-M3 FAISS index (737K movies, 1024-dim embeddings).

**Approach:** For each user, average the embeddings of their top-rated movies (≥ 4.0)
to form a "user profile vector", then FAISS search for nearest movies. This is pure semantic retrieval, no collaborative signal.

In [36]:
import faiss
from tqdm import tqdm

res = faiss.StandardGpuResources()
res.setTempMemory(2 * 1024 * 1024 * 1024)
faiss_index_gpu = faiss.index_cpu_to_gpu(res, 0, faiss_index)
print("Index on GPU")

# Pre-cache train lookups
train_by_user = {uid: grp for uid, grp in train.groupby('userId')}

# Pre-cache all movie embeddings
print("Pre-reconstructing movie embeddings...")
movie_vecs = {}
for mid, rowid in tqdm(movieid_to_rowid.items()):
    try:
        movie_vecs[mid] = faiss_index.reconstruct(int(rowid))  # CPU index for reconstruct
    except RuntimeError:
        pass
print(f"Cached {len(movie_vecs):,} movie vectors")

# Updated profile builder using caches
def get_user_profile_fast(user_id: int, top_n: int = 15) -> np.ndarray | None:
    if user_id not in train_user_items:
        return None
    user_train = train_by_user[user_id].sort_values('rating', ascending=False)
    liked = user_train[user_train['rating'] >= 4.0].head(top_n)

    if liked.empty:
        liked = user_train.head(top_n)

    vecs, weights = [], []
    for row in liked.itertuples():
        if row.movieId in movie_vecs:
            vecs.append(movie_vecs[row.movieId])
            weights.append(row.rating)  # weight by rating

    # In get_user_profile_fast:
    if not vecs:
        return None

    profile = np.mean(vecs, axis=0).astype('float32').reshape(1, -1)  # ← here
    faiss.normalize_L2(profile)
    return profile

    # weights = np.array(weights, dtype='float32')
    # weights /= weights.sum()  # normalize weights
    # profile = np.average(vecs, axis=0, weights=weights).astype('float32').reshape(1, -1)
    # faiss.normalize_L2(profile)
    # return profile


print("Building user profile matrix...")
uids_ordered = list(eval_users)
profiles, uids_with_profile = [], []

for uid in tqdm(uids_ordered):
    p = get_user_profile_fast(uid)
    if p is not None:
        profiles.append(p)
        uids_with_profile.append(uid)

profile_matrix = np.vstack(profiles).astype('float32')  # shape: (N_users, dim)
print(f"Profile matrix: {profile_matrix.shape}")

K_fetch = K + 500
print(f"Batch searching {len(uids_with_profile):,} users on GPU...")
t0 = time.time()
D_all, I_all = faiss_index_gpu.search(profile_matrix, K_fetch)
print(f"Done in {time.time() - t0:.1f}s")

print("Building recommendation cache...")
uid_to_recs = {}

for i, uid in enumerate(uids_with_profile):
    seen = train_user_items.get(uid, set())
    recs = []
    for row_id in I_all[i]:
        if row_id < 0:
            continue
        mid = rowid_to_movieid.get(int(row_id))
        if mid and mid not in seen:
            recs.append(mid)
            if len(recs) >= K:
                break
    uid_to_recs[uid] = recs

# Fallback for users without a profile
for uid in uids_ordered:
    if uid not in uid_to_recs:
        uid_to_recs[uid] = popularity_recommend(uid, K)

print(f"Recs cached for {len(uid_to_recs):,} users")

def faiss_recommend_fast(user_id: int, k: int = K) -> list:
    return uid_to_recs.get(user_id, [])[:k]

# Preview recs for a sample of users
sample_users = uids_ordered[1:2]  # change to any user IDs you want

print("\nSample Recommendations:")
for uid in sample_users:
    recs = faiss_recommend_fast(uid)
    rec_titles = movies[movies['movieId'].isin(recs)][['movieId', 'title']].set_index('movieId')
    # preserve ranked order
    rec_titles = rec_titles.reindex(recs).reset_index()
    print(f"\nUser {uid} (showing top {len(recs)}):")
    for rank, row in enumerate(rec_titles.itertuples(), 1):
        print(f"  {rank:2}. {row.title}")

all_results.append(evaluate_model(faiss_recommend_fast, 'Content-Based (FAISS)'))

Index on GPU
Pre-reconstructing movie embeddings...


100%|██████████| 71380/71380 [00:00<00:00, 226212.76it/s]


Cached 71,380 movie vectors
Building user profile matrix...


100%|██████████| 5000/5000 [00:05<00:00, 879.24it/s]


Profile matrix: (5000, 1024)
Batch searching 5,000 users on GPU...
Done in 0.5s
Building recommendation cache...
Recs cached for 5,000 users

Sample Recommendations:

User 106593 (showing top 20):
   1. The Predator (2018)
   2. Predator (1987)
   3. Star Trek (2009)
   4. The Incident (2014)
   5. Hercules (2014)
   6. Alien Predator (Mutant II) (Falling, The) (1985)
   7. Hijacked (2012)
   8. Astro (2018)
   9. Unknown (2011)
  10. Diablo (2015)
  11. Fantastic Four (2015)
  12. Predestination (2014)
  13. Interceptors (1999)
  14. Dreamcatcher (2003)
  15. Star Trek: Insurrection (1998)
  16. Predators (2010)
  17. Revolt (2017)
  18. Skyline (2010)
  19. Stalker (1979)
  20. Titan A.E. (2000)
Content-Based (FAISS): P@20=0.0113  NDCG@20=0.0125  HR@20=0.1550  (0.2s)


In [35]:
# Run this to understand what's actually happening
uid = uids_ordered[0]
user_train = train_by_user[uid]
print(f"User {uid}: {len(user_train)} total ratings, {(user_train['rating']>=4.0).sum()} liked")

# Check how many eval users have very few ratings
rating_counts = [len(train_by_user[uid]) for uid in eval_users if uid in train_by_user]
print(f"Median ratings per eval user: {np.median(rating_counts):.0f}")
print(f"Users with <5 ratings: {sum(1 for c in rating_counts if c < 5)}")

User 190163: 365 total ratings, 195 liked
Median ratings per eval user: 289
Users with <5 ratings: 0


## Baseline 4: FAISS + BGE-M3 Reranker (Two-Stage)
**Stage 1:** FAISS retrieves top-1000 candidates via semantic search.  
**Stage 2:** BGE-M3 multi-granularity model reranks using **dense + sparse + ColBERT** scores.

In [15]:
import transformers.utils.import_utils as import_utils
if not hasattr(import_utils, 'is_torch_fx_available'):
    import_utils.is_torch_fx_available = lambda: False
from FlagEmbedding import BGEM3FlagModel

In [ ]:
if ENABLE_RERANKER:
    def build_user_query(user_id: int, top_n: int = 5) -> str:
        user_train = train_by_user[user_id].sort_values('rating', ascending=False)
        liked = user_train[user_train['rating'] >= 3.5].head(top_n)
        if liked.empty:
            liked = user_train.head(top_n)
        titles = liked.merge(movies[['movieId', 'title']], on='movieId')['title'].tolist()
        return 'Movies similar to: ' + ', '.join(titles)

    # ── Stage 1: Build candidate pools ──
    print("Building rerank candidate pools...")
    uid_to_candidates = {}
    for i, uid in enumerate(tqdm(uids_with_profile)):
        seen = train_user_items.get(uid, set())
        candidates = []
        for row_id in I_all[i]:
            if row_id < 0:
                continue
            mid = rowid_to_movieid.get(int(row_id))
            if mid and mid not in seen:
                candidates.append((int(row_id), mid))
            if len(candidates) >= RERANK_CANDIDATES:
                break
        uid_to_candidates[uid] = candidates

    # ── SPEEDUP 1: Deduplicate docs — many movies appear across multiple users ──
    # Encode each unique doc only once instead of once per user occurrence
    print("Collecting unique docs and queries...")
    all_unique_docs = list({rowid_to_doc.get(rid, '') for uid in uids_with_profile
                            for rid, _ in uid_to_candidates.get(uid, [])})
    doc_to_idx = {doc: i for i, doc in enumerate(all_unique_docs)}

    all_queries = {uid: build_user_query(uid) for uid in uids_with_profile}
    all_unique_queries = list(set(all_queries.values()))
    query_to_idx = {q: i for i, q in enumerate(all_unique_queries)}

    print(f"Unique docs: {len(all_unique_docs):,}  |  Unique queries: {len(all_unique_queries):,}")
    # vs total pairs before: {len(uids_with_profile) * RERANK_CANDIDATES:,}

    # ── SPEEDUP 2: Encode docs & queries separately in large batches ──
    # Much faster than pairwise compute_score — O(D+Q) encodes vs O(D*Q)
    ENCODE_BATCH = 2048  # A100 handles this easily

    print("Encoding all unique docs...")
    t0 = time.time()
    doc_embeddings = bge_m3.encode(
        all_unique_docs,
        batch_size=ENCODE_BATCH,
        max_length=512,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,  # SPEEDUP 3: skip ColBERT (most expensive, ~40% of time)
    )
    print(f"Docs encoded in {time.time()-t0:.1f}s")

    print("Encoding all unique queries...")
    t0 = time.time()
    query_embeddings = bge_m3.encode(
        all_unique_queries,
        batch_size=ENCODE_BATCH,
        max_length=128,  # queries are short
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    print(f"Queries encoded in {time.time()-t0:.1f}s")

    # ── SPEEDUP 4: Score via dot product (no ColBERT) ──
    # dense: matrix multiply  |  sparse: dot product
    dense_doc   = doc_embeddings['dense_vecs']    # (n_docs, dim)
    dense_query = query_embeddings['dense_vecs']  # (n_queries, dim)
    sparse_doc  = doc_embeddings['lexical_weights']
    sparse_query = query_embeddings['lexical_weights']

    DENSE_W, SPARSE_W = 0.6, 0.4  # rebalanced since no ColBERT

    print("Computing dense scores via matrix multiply...")
    dense_scores_matrix = dense_query @ dense_doc.T  # (n_queries, n_docs)

    # ── Stage 3: Build reranked recs ──
    print("Building reranked recommendation cache...")
    uid_to_reranked_recs = {}
    for uid in tqdm(uids_with_profile):
        candidates = uid_to_candidates.get(uid, [])
        if not candidates:
            uid_to_reranked_recs[uid] = popularity_recommend(uid, K)
            continue

        q_idx = query_to_idx[all_queries[uid]]
        scores = []
        for rid, mid in candidates:
            doc = rowid_to_doc.get(rid, '')
            d_idx = doc_to_idx[doc]

            dense  = float(dense_scores_matrix[q_idx, d_idx])

            # Sparse dot product
            sq, sd = sparse_query[q_idx], sparse_doc[d_idx]
            sparse = sum(sq.get(tok, 0) * w for tok, w in sd.items())

            scores.append((DENSE_W * dense + SPARSE_W * sparse, mid))

        uid_to_reranked_recs[uid] = [mid for _, mid in sorted(scores, reverse=True)[:K]]

    for uid in uids_ordered:
        if uid not in uid_to_reranked_recs:
            uid_to_reranked_recs[uid] = popularity_recommend(uid, K)

    def faiss_rerank_recommend_fast(user_id: int, k: int = K) -> list:
        return uid_to_reranked_recs.get(user_id, [])[:k]

    all_results.append(evaluate_model(faiss_rerank_recommend_fast, 'FAISS + BGE-M3 Reranker'))

Building rerank candidate pools...


100%|██████████| 5000/5000 [00:00<00:00, 5972.87it/s]


Unique docs: 9,623  |  Unique queries: 4,999
Encoding all unique docs...


Inference Embeddings: 100%|██████████| 5/5 [00:09<00:00,  1.94s/it]


Docs encoded in 10.9s
Encoding all unique queries...


Inference Embeddings: 100%|██████████| 3/3 [00:02<00:00,  1.33it/s]


Queries encoded in 5.9s
Computing dense scores via matrix multiply...


## Results: Baseline Comparison

In [ ]:
results_df = pd.DataFrame(all_results)
metric_cols = [c for c in results_df.columns if c not in ('Model', 'Coverage', 'Eval_sec')]


print('║              BASELINE COMPARISON — CineMatch                ║\n')
display(results_df.set_index('Model').style.format({
    c: '{:.4f}' for c in metric_cols
}).format({
    'Coverage': '{:,.0f}',
    'Eval_sec': '{:.1f}',
}).background_gradient(subset=metric_cols, cmap='YlGn'))

# Bar Chart: Key Metrics Side by Side
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
colors = ['#95a5a6', '#3498db', '#e74c3c', '#2ecc71'][:len(results_df)]

for ax, col in zip(axes, [f'NDCG@{K}', f'HitRate@{K}', f'Precision@{K}', f'MRR@{K}']):
    bars = ax.bar(results_df['Model'], results_df[col], color=colors, alpha=0.9, edgecolor='white', linewidth=1.5)
    ax.set_title(col, fontsize=14, fontweight='bold')
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=35)
    # Value labels on bars
    for bar, val in zip(bars, results_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle(f'CineMatch Baseline Comparison (Top-{K}, {len(eval_users):,} eval users)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Improvement Summary
if len(results_df) >= 2:
    best = results_df.loc[results_df[f'NDCG@{K}'].idxmax()]
    naive = results_df.iloc[0]  # Global Popularity
    print(f'\n Best model: {best["Model"]}')
    print(f'   NDCG@{K} improvement over naive: {((best[f"NDCG@{K}"] / naive[f"NDCG@{K}"]) - 1) * 100:+.1f}%')
    print(f'   HitRate@{K} improvement over naive: {((best[f"HitRate@{K}"] / naive[f"HitRate@{K}"]) - 1) * 100:+.1f}%')

## Cross-Cultural Evaluation
Evaluate specifically on users who **rate non-English movies**. This shows which baselines handle cross-cultural recommendations and which fail.

In [ ]:
# Identify cross-cultural eval users
# Users whose test set contains at least one non-English movie

# Build movieId → language lookup
tmdb_lang = pd.read_csv(
    DATA_DIR / 'TMDB_movie_dataset_v11.csv',
    usecols=['id', 'original_language'],
    low_memory=False,
)
tmdb_lang['id'] = pd.to_numeric(tmdb_lang['id'], errors='coerce')
tmdb_lang = tmdb_lang.dropna(subset=['id'])
tmdb_lang['id'] = tmdb_lang['id'].astype(int)

# Map movieId → tmdbId → language
links_lang = links[['movieId', 'tmdbId']].dropna().copy()
links_lang['tmdbId'] = links_lang['tmdbId'].astype(int)
links_lang = links_lang.merge(tmdb_lang.rename(columns={'id': 'tmdbId'}), on='tmdbId', how='inner')
movieid_to_lang = dict(zip(links_lang['movieId'], links_lang['original_language']))

# Find eval users who have non-EN test items
cross_cultural_users = []
for uid in eval_users:
    test_items = test_user_items[uid]
    langs = {movieid_to_lang.get(m, 'unknown') for m in test_items}
    if langs - {'en', 'unknown'}:  # has at least one non-EN movie
        cross_cultural_users.append(uid)

print(f'Cross-cultural eval users: {len(cross_cultural_users)} / {len(eval_users)}')

if len(cross_cultural_users) >= 50:
    # Re-evaluate all models on cross-cultural subset
    original_eval = eval_users.copy()
    eval_users_backup = eval_users
    eval_users = np.array(cross_cultural_users)

    cc_results = []
    models_to_eval = [
        ('Global Popularity', popularity_recommend),
        ('ALS Collaborative Filtering', als_recommend),
        ('Content-Based (FAISS)', faiss_recommend),
    ]
    # Add reranker if available
    if 'faiss_rerank_recommend' in dir():
        models_to_eval.append(('FAISS + BGE-M3 Reranker', faiss_rerank_recommend))

    print('\n═══ CROSS-CULTURAL EVALUATION ═══')
    for name, fn in models_to_eval:
        cc_results.append(evaluate_model(fn, name))

    cc_df = pd.DataFrame(cc_results)
    display(cc_df.set_index('Model').style.format({
        c: '{:.4f}' for c in metric_cols
    }).background_gradient(subset=metric_cols, cmap='YlGn'))

    # Restore original eval users
    eval_users = eval_users_backup
else:
    print('Not enough cross-cultural users for meaningful evaluation.')

## IMDB FAISS Retrieval + BGE-M3 Reranking (Cross-Cultural Bias)

In [ ]:
from __future__ import annotations
import json, time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = Path('/content/drive/MyDrive/cinematch')
except ImportError:
    _here = Path.cwd()
    for _c in [_here, *_here.parents]:
        if (_c / 'Data').exists() and (_c / 'src').exists():
            DATA_DIR = _c / 'Data'; break

IMDB_FAISS_PATH   = DATA_DIR / 'outputs' / 'imdb' / 'imdb_movies_bge_m3_flatip.faiss'
IMDB_META_PATH    = DATA_DIR / 'outputs' / 'imdb' / 'imdb_movies_meta.csv'
IMDB_CATALOG_PATH = DATA_DIR / 'outputs' / 'imdb' / 'imdb_movies_catalog.csv'

# Language bias config
PREFERRED_LANGS = ['te', 'hi', 'ta', 'ko', 'ja']  # Telugu, Hindi, Tamil, Korean, Japanese
PREFERRED_BOOST = 0.15    # additive boost to rerank score for preferred-language films
NON_EN_BOOST    = 0.05    # smaller boost for any non-English film
FAISS_CANDIDATES = 3000    # retrieve this many from FAISS before reranking
TOP_K           = 15      # final number of recommendations

# LOAD INDEX & DATA
print('Loading IMDB FAISS index...')
t0 = time.time()
index = faiss.read_index(str(IMDB_FAISS_PATH))
print(f'  Loaded {index.ntotal:,} vectors (dim={index.d}) in {time.time()-t0:.1f}s')

meta = pd.read_csv(IMDB_META_PATH)
catalog = pd.read_csv(IMDB_CATALOG_PATH, usecols=['row_id', 'movieDoc'])
meta = meta.merge(catalog, on='row_id', how='left')
meta['movieDoc'] = meta['movieDoc'].fillna('').astype(str)
meta['tmdb_original_language'] = meta['tmdb_original_language'].fillna('unknown')

# Fast lookups
rowid_to_meta = meta.set_index('row_id').to_dict('index')
print(f'  Meta loaded: {len(meta):,} movies')
print(f'  Lang distribution in index:')
for lang in PREFERRED_LANGS:
    n = (meta['tmdb_original_language'] == lang).sum()
    print(f'    {lang}: {n:,}')
en_count = (meta['tmdb_original_language'] == 'en').sum()
print(f'    en: {en_count:,}')


print('\nLoading BGE-M3 reranker...')
try:
    from FlagEmbedding import BGEM3FlagModel
    reranker = BGEM3FlagModel('BAAI/bge-m3-unsupervised', use_fp16=True)
    HAS_RERANKER = True
    print('  BGE-M3 loaded successfully.')
except ImportError:
    print('  FlagEmbedding not installed — using FAISS scores only (no reranking).')
    print('  Install with: pip install FlagEmbedding')
    HAS_RERANKER = False

# SEARCH FUNCTION
def search_imdb(
    query: str,
    top_k: int = TOP_K,
    preferred_langs: list[str] = PREFERRED_LANGS,
    preferred_boost: float = PREFERRED_BOOST,
    non_en_boost: float = NON_EN_BOOST,
    n_candidates: int = FAISS_CANDIDATES,
    rerank: bool = True,
    rerank_weights: list[float] = [0.4, 0.2, 0.4],  # [dense, sparse, colbert]
) -> pd.DataFrame:
    """
    Two-stage IMDB movie search with cross-cultural bias.

    Stage 1: FAISS semantic retrieval (BGE-M3 embeddings, IndexFlatIP cosine)
    Stage 2: BGE-M3 multi-granularity reranking (dense + sparse + ColBERT)
    + Language bias: preferred_boost for target langs, non_en_boost for any non-EN

    Args:
        query: Natural language query (e.g., "emotional family drama Telugu")
        top_k: Number of results to return
        preferred_langs: ISO language codes to boost
        preferred_boost: Additive score boost for preferred languages
        non_en_boost: Additive score boost for any non-English film
        n_candidates: Number of FAISS candidates before reranking
        rerank: Whether to apply BGE-M3 reranking (requires FlagEmbedding)
        rerank_weights: [dense, sparse, colbert] weight mix
    """
    t0 = time.time()

    # FAISS retrieval
    qvec = reranker.encode([query], batch_size=64)['dense_vecs'] if HAS_RERANKER else None
    if qvec is None:
        # Fallback: use sentence-transformers directly
        from sentence_transformers import SentenceTransformer
        _encoder = SentenceTransformer('BAAI/bge-m3-unsupervised')
        qvec = _encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)

    qvec = qvec.astype('float32')
    faiss.normalize_L2(qvec)
    D, I = index.search(qvec, n_candidates)

    candidates = []
    for score, row_id in zip(D[0], I[0]):
        if row_id < 0:
            continue
        info = rowid_to_meta.get(int(row_id), {})
        candidates.append({
            'row_id': int(row_id),
            'faiss_score': float(score),
            'title': info.get('primaryTitle', '?'),
            'year': info.get('startYear', ''),
            'genres': info.get('genres', ''),
            'lang': info.get('tmdb_original_language', 'unknown'),
            'lang_bucket': info.get('lang_bucket', 'unknown'),
            'rating': info.get('averageRating', None),
            'votes': info.get('numVotes', None),
            'movieDoc': info.get('movieDoc', ''),
        })

    if not candidates:
        print('No candidates found.')
        return pd.DataFrame()

    cdf = pd.DataFrame(candidates)

    # BGE-M3 Reranking
    if rerank and HAS_RERANKER and len(cdf) > 0:
        pairs = [[query, doc] for doc in cdf['movieDoc'].tolist()]
        scores = reranker.compute_score(pairs, weights_for_different_modes=rerank_weights)
        if isinstance(scores, dict):
            scores = scores.get('colbert+sparse+dense', list(scores.values())[0])
        if not isinstance(scores, list):
            scores = [scores]
        cdf['rerank_score'] = scores
    else:
        cdf['rerank_score'] = cdf['faiss_score']

    # Language bias
    cdf['lang_boost'] = 0.0
    cdf.loc[cdf['lang'].isin(preferred_langs), 'lang_boost'] = preferred_boost
    cdf.loc[(cdf['lang'] != 'en') & (cdf['lang_boost'] == 0), 'lang_boost'] = non_en_boost
    cdf['final_score'] = cdf['rerank_score'] + cdf['lang_boost']

    # Sort & return top-K
    cdf = cdf.sort_values('final_score', ascending=False).head(top_k).reset_index(drop=True)
    cdf.index = range(1, len(cdf) + 1)
    cdf.index.name = 'rank'

    elapsed = time.time() - t0
    print(f'Search completed in {elapsed:.2f}s  |  Reranked: {rerank and HAS_RERANKER}  |  Candidates: {len(candidates)}')

    display_cols = ['title', 'year', 'lang', 'genres', 'rating', 'votes',
                    'faiss_score', 'rerank_score', 'lang_boost', 'final_score']
    return cdf[display_cols]



print('  DEMO: Cross-Cultural IMDB Search with Language Bias')

queries = [
    'emotional family drama with sacrifice and redemption',
    'action thriller with car chases and heist',
    'romantic comedy set in college campus',
    'mythological epic with gods and demons war',
    'psychological horror supernatural mystery village',
    'action films with a dominant, strategic anti-hero or undercover protagonist' #saaho
]

for q in queries:
    print(f'\n{"─"*70}')
    print(f'  Query: "{q}"')
    print(f'  Preferred langs: {PREFERRED_LANGS} (boost={PREFERRED_BOOST})')
    print(f'{"─"*70}')
    result = search_imdb(q, top_k=TOP_K)
    display(result)

    # Show language breakdown
    lang_counts = result['lang'].value_counts()
    non_en = (result['lang'] != 'en').sum()
    pref = result['lang'].isin(PREFERRED_LANGS).sum()
    print(f' Non-EN: {non_en}/{TOP_K} | Preferred lang: {pref}/{TOP_K} | Langs: {dict(lang_counts)}')